In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid


In [2]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site
nodesReq = 1
coresReq = 2
ramReq = 8

# we scale up the requirements a bit to account for the potential of others joining the selected site. 
totalCoreAvail = nodesReq * coresReq * 1.2
totalRamAvail = nodesReq * ramReq * 1.2

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(usableSite)

['MICH', 'BRIST', 'UCSD', 'MASS', 'EDUKY', 'CERN', 'FIU', 'SRI', 'UTAH', 'GATECH', 'STAR', 'DALL', 'LOSA', 'WASH', 'SALT', 'TACC', 'KANS', 'AMST', 'CLEM', 'NCSA', 'RUTG', 'INDI', 'PRIN', 'NEWY', 'PSC', 'HAWI', 'TOKY', 'GPN', 'ATLA', 'SEAT', 'MAX', 'EDC']


In [3]:
import random
siteName = random.choice(usableSite)
sliceName = "Swarmy"
print(siteName)

slice = fablib.new_slice(name=sliceName)

for i in range(1, nodesReq + 1):
    node = slice.add_node(name=f"node{i}", 
                          site=siteName,
                          cores=coresReq,
                          ram=ramReq,
                          disk=30, 
                          image='default_ubuntu_22')
    
slice.submit()   


Retry: 8, Time: 182 sec


ID,04efc959-43d6-4a7e-b960-7b3b434d95ed
Name,Swarmy
Lease Expiration (UTC),2026-04-29 21:10:34 +0000
Lease Start (UTC),2026-04-28 21:10:34 +0000
Project ID,8b6dfc51-02ad-4f00-a389-6e75d8b61a26
State,StableOK
Email,lngo@wcupa.edu
UserId,8eecd713-fa8f-4b3b-8883-1ff9b021fa53


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
3103dcf2-a556-47e3-a344-717536f8deab,node1,2,8,100,default_ubuntu_22,qcow2,psc-w1.fabric-testbed.net,PSC,ubuntu,None,Active,,ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@None,/home/fabric/.ssh/slice_key.pub,/home/fabric/.ssh/slice_key


'04efc959-43d6-4a7e-b960-7b3b434d95ed'

In [4]:
import time

while True:
    time.sleep(10)
    slice.update()
    print("Slice state:", slice.get_state())
    print("Slice stable:", slice.isStable())
    if node.get_management_ip() != None:
        for node in slice.get_nodes():
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

Slice state: StableOK
Slice stable: True
---- node1 ----
reservation state: Active
management ip: 2001:5e8:ff00:ffff:f816:3eff:fec2:71d6
username: ubuntu
error: 
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:5e8:ff00:ffff:f816:3eff:fec2:71d6


In [5]:
from pathlib import Path

nodes = slice.get_nodes()
private_key = fablib.get_default_slice_key()["slice_private_key_file"]
ssh_config = "/home/fabric/work/fabric_config/ssh_config"

inventory = """all:
  children:
    fabric_nodes:
      hosts:
"""

for node in nodes:
    inventory += f"""        {node.get_name()}:
          ansible_host: "{node.get_management_ip()}"
          ansible_user: "{node.get_username()}"
          ansible_ssh_private_key_file: "{private_key}"
          ansible_ssh_common_args: "-F {ssh_config}"
          ansible_python_interpreter: "/usr/bin/python3"
"""

Path("inventory.yml").write_text(inventory)
print(inventory)

all:
  children:
    fabric_nodes:
      hosts:
        node1:
          ansible_host: "2001:5e8:ff00:ffff:f816:3eff:fec2:71d6"
          ansible_user: "ubuntu"
          ansible_ssh_private_key_file: "/home/fabric/.ssh/slice_key"
          ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
          ansible_python_interpreter: "/usr/bin/python3"



In [6]:
!ansible all -i inventory.yml -m ping

node1 | SUCCESS => {
    "changed": false,
    "ping": "pong"
}


In [8]:
!ansible-playbook -i inventory.yml install-container-engines.yml


PLAY [Install Podman, Apptainer, containerd, and nerdctl on FABRIC Ubuntu nodes] ***

TASK [Gathering Facts] *********************************************************
ok: [node1]

TASK [Update apt cache] ********************************************************
changed: [node1]

TASK [Install base packages, Podman, containerd, runc, and CNI plugins] ********
changed: [node1]

TASK [Add Apptainer Ubuntu PPA] ************************************************
changed: [node1]

TASK [Install Apptainer] *******************************************************
changed: [node1]

TASK [Ensure containerd configuration directory exists] ************************
changed: [node1]

TASK [Generate default containerd configuration if missing] ********************
changed: [node1]

TASK [Use systemd cgroup driver in containerd config] **************************
changed: [node1]

TASK [Enable and start containerd] *********************************************
ok: [node1]

TASK [Install nerdctl CLI for co

In [9]:
!ansible-playbook -i inventory.yml demo-container-engines.yml


PLAY [Demonstrate Podman, Apptainer, and containerd on FABRIC nodes] ***********

TASK [Podman demo] *************************************************************
ok: [node1]

TASK [Show Podman output] ******************************************************
ok: [node1] => {
    "podman_demo.stdout_lines": [
        "Podman demo",
        "hostname=node1",
        "uid=0 gid=0",
        "3.20.10"
    ]
}

TASK [Apptainer demo using a Docker/OCI image] *********************************
ok: [node1]

TASK [Show Apptainer output] ***************************************************
ok: [node1] => {
    "apptainer_demo.stdout_lines": [
        "Apptainer demo",
        "hostname=node1",
        "uid=0 gid=0",
        "3.20.10"
    ]
}

TASK [Pull Alpine into containerd content store using ctr] *********************
ok: [node1]

TASK [Run Alpine directly with containerd ctr] *********************************
ok: [node1]

TASK [Show ctr output] **************************************************